# Visualização exploratória e previsão-baseline do estoque inscrito e da arrecadação mensal a partir de um dump estático de pesquisa

**Exploratory visualization and baseline forecasting of monthly tax-debt stock and collection from a static research dump**

## Resumo

Este notebook documenta, de forma reproduzível, (i) a análise exploratória do co-movimento entre **estoque inscrito** (`valor_sem_honorarios`) e **arrecadação** (`valor_total`) no painel mensal derivado do dump estático da API Inteligência Fiscal (extração congelada `extracao=2026-03`) e (ii) um protocolo *walk-forward* (holdout $\approx$ 12 meses) que ranqueia baselines e modelos leves: naive, sazonal, OLS/Ridge, SARIMAX, HistGradientBoosting, Prophet e MLP-2 (PyTorch). Métricas de acurácia: **MAPE**, **sMAPE** e **RMSE** (Hyndman & Athanasopoulos, 2021, FPP3 §5.8). Figuras em **Plotly**: interativas via `fig.show()` e estáticas (PNG) via `fig.write_image` (Kaleido) para publicação. Intervalos de previsão: nativos (SARIMAX / Prophet) ou **empíricos** por quantis de resíduos OOS (HGB / MLP — sem cobertura nominal garantida). Escopo *toy*/baseline de pesquisa: tipagem frágil, ausência de garantias e de dívida não inscrita limitam interpretação causal. Conteúdo, experimentos e conclusões são do autor; a formatação do texto teve assistência de IA.

## Abstract (EN)

We explore monthly registered tax-debt stock and collection from a frozen Inteligência Fiscal dump (`extracao=2026-03`), with Plotly EDA (interactive + Kaleido static export) and walk-forward baselines (naive, seasonal, OLS/Ridge, tuned SARIMAX, HGB, Prophet, 2-layer MLP). Metrics: MAPE, sMAPE, RMSE. Scope is reproducible research—not operational realtime forecasting.

## Palavras-chave / Keywords

estoque inscrito; arrecadação; GARE; série temporal; walk-forward; sMAPE; SARIMAX; Prophet; MLP; HistGradientBoosting; dump estático; FPP3; Plotly; Kaleido


## 1. Introdução

### Dump de pesquisa versus sistemas operacionais

A API Inteligência Fiscal expõe um **dump estático** para pesquisa: um retrato versionado por `extracao`, não um feed operacional. Congelar a extração e preferir Parquet local garante que a revisão do notebook não dependa da API estar disponível.

### Conceitos jurídico-tributários (para leitor de métodos)

| Conceito | Leitura operacional neste artefato |
|---|---|
| **Crédito tributário** | Obrigação pecuniária constituída perante o Fisco. |
| **Inscrição em dívida ativa** | Formaliza o crédito inadimplido; o estoque mensal espelha o universo **já inscrito**. |
| **CDA** | Certidão de Dívida Ativa — título do crédito inscrito. |
| **Estoque inscrito (stock)** | Foto agregada do valor inscrito (`valor_sem_honorarios`). |
| **Arrecadação / fluxo (flow)** | Valor arrecadado no mês (`valor_total`) — **alvo $y$** do forecast. |
| **GARE** | Guia de Arrecadação de Receitas Estaduais. |
| **Honorários** | Componentes acessórios; estoque analítico usa `valor_sem_honorarios`. |
| **Estoque vs fluxo** | Co-movimento **não** é identidade contábil nem prova causal. |
| **Dívida não inscrita** | **Ausente** deste dump — lacuna estrutural. |
| **Ajuizamento / execução fiscal** | Fora do join deste notebook. |

### Definições analíticas (primeira aparição — fórmulas e âncoras)

#### Walk-forward / time-series cross-validation

Avaliação em que cada previsão usa apenas informação **anterior** ao instante previsto, evitando vazamento temporal (Hyndman & Athanasopoulos, 2021, FPP3 §5.10).

**Aplicação aqui.** Série mensal de comprimento $n$; holdout dos últimos $h=12$ meses; treino = índices $t < n-h$. Em modelos tabulares (OLS/Ridge/HGB), a cada passo $t$ do holdout o refit usa somente $t' < t$ (*one-step walk-forward*).

**Limitações.** Um único bloco de holdout não é CV completa com múltiplas origens; $h=12$ é pequeno para inferência sobre cobertura de intervalos.

#### Naive lag-1 e sazonal lag-12

Baselines simples (FPP3 §5.2):

$$
\hat{y}_{t}^{\mathrm{naive}} = y_{t-1}, \qquad
\hat{y}_{t}^{\mathrm{saz}} = y_{t-12}.
$$

**Aplicação aqui.** Modelos `M0_naive_lag1` e `M0_naive_sazonal_lag12` obrigatórios antes de modelos mais ricos.

**Limitações.** Ignoram covariáveis e mudanças de regime; o sazonal assume periodicidade anual estável.

#### MAPE (Mean Absolute Percentage Error)

$$
\mathrm{MAPE} = \frac{1}{m}\sum_{i=1}^{m}\left|\frac{y_i-\hat{y}_i}{y_i}\right|
\quad\text{(observações com } y_i=0 \text{ excluídas)}.
$$

Âncora: FPP3 §5.8; discussão clássica de medidas percentuais em Armstrong & Collopy (1992) e nas competições M (Makridakis et al.).

**Aplicação aqui.** Holdout $h\approx 12$, $y=$ `valor_total`. Ranking primário por MAPE.

**Limitações.** Indefinido/instável se $y\approx 0$; assimétrico em sub vs superprevisão; não é a única medida de ranking.

#### sMAPE (symmetric MAPE)

$$
\mathrm{sMAPE} = \frac{1}{m}\sum_{i=1}^{m}\frac{2|y_i-\hat{y}_i|}{|y_i|+|\hat{y}_i|}
\quad\text{(se } y_i=\hat{y}_i=0\text{, contribuição }0\text{)}.
$$

Usso nas competições M / FPP3 §5.8 (forma simétrica).

**Aplicação aqui.** Complementa MAPE; ordenação secundária das barras de comparação.

**Limitações.** Ainda percentual; interpretação muda se denominador for dominado por $\hat{y}$.

#### RMSE (Root Mean Squared Error)

$$
\mathrm{RMSE} = \sqrt{\frac{1}{m}\sum_{i=1}^{m}(y_i-\hat{y}_i)^2}.
$$

**Aplicação aqui.** Mesma janela de holdout; unidade = R$ (mesma de $y$).

**Limitações.** Sensível a outliers; não é adimensional — não comparar RMSE entre séries de escalas distintas.

#### Outliers IQR (Tukey)

Para uma margem $x$, com quartis $Q_1,Q_3$ e $\mathrm{IQR}=Q_3-Q_1$:

$$
x \notin \bigl[Q_1-1{,}5\cdot\mathrm{IQR},\; Q_3+1{,}5\cdot\mathrm{IQR}\bigr].
$$

Âncora: Tukey (1977), *Exploratory Data Analysis*.

**Aplicação aqui.** Flag descritivo se outlier em estoque **ou** arrecadação no scatter.

**Limitações.** Regra univariada; **não** implica anomalia jurídica nem causalidade.

#### Dual-axis (caveat)

Estoque e fluxo mensal diferem por ordens de magnitude; um único eixo Y comprime a arrecadação. Dual-axis (Y esquerdo = estoque, Y direito = arrecadação) é um **encoding de escala**, não prova de comensurabilidade.

**O que não afirmar.** Comparar magnitudes cruzadas entre eixos; causalidade estoque→arrecadação a partir do overlay.

#### Intervalos de previsão: nativo vs empírico

- **Nativo:** produzido pelo modelo (SARIMAX `summary_frame`; Prophet `yhat_lower`/`yhat_upper`) sob suas hipóteses (FPP3 §5.5).
- **Empírico:** $\hat{y}_t + q_{\alpha}(e^{\mathrm{OOS}})$, quantis dos resíduos *out-of-sample* (HGB/MLP).

**Limitações.** PI empírico **não** garante cobertura nominal $1-\alpha$; não misturar leituras de cobertura.

### Perguntas de pesquisa

- **RQ1.** Como estoque e arrecadação co-movimentam-se (níveis, sazonalidade, outliers, composição ICMS)?
- **RQ2.** Qual modelo (M0–M5) melhor prevê `valor_total` no holdout $\approx$ 12m (MAPE / sMAPE / RMSE)?


## 2. Dados e congelamento da extração

Carregamos settings via `.env` (`DUMP_ROOT`, `EXTRACAO_*`, API). Preferimos o Parquet local `painel_mensal_estoque_arrecadacao.parquet`; se ausente, reconstruímos pelas séries agregadas da API. Paths absolutos do HD **não** são impressos nos outputs publicáveis. Figuras estáticas (Kaleido) são gravadas em `projects/monitoramento/output/figures/`.


In [1]:
from pathlib import Path
import os, sys, warnings, time
warnings.filterwarnings("ignore")

# Resolve mono from this notebook location (no hard-coded home in logic).
NB_DIR = Path.cwd()
candidates = [NB_DIR, *NB_DIR.parents]
MONO = next((p for p in candidates if (p / "shared" / "cemepi_api").exists()), NB_DIR)
os.chdir(MONO)
sys.path.insert(0, str(MONO))

from shared.cemepi_api import CemepiClient, load_settings

S = load_settings()
S.ano, S.mes = int(os.getenv("EXTRACAO_ANO", S.ano)), int(os.getenv("EXTRACAO_MES", S.mes))
# freeze
S.ano, S.mes = 2026, 3
C = CemepiClient(S)
EXTRACAO = f"{S.ano:04d}-{S.mes:02d}"
assert EXTRACAO == "2026-03"

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)
SEED = 42
np.random.seed(SEED)

FIG_DIR = MONO / "projects/monitoramento/output/figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, stem: str, width: int = 1000, height: int = 520) -> Path:
    """Interactive already via fig.show(); static PNG via Kaleido for publication."""
    out = FIG_DIR / f"{stem}.png"
    try:
        fig.write_image(str(out), width=width, height=height, scale=2)
        print("static ->", public_path_label(out))
    except Exception as e:
        print("Kaleido write_image skip:", type(e).__name__, e)
    return out

def first_list(obj):
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        for v in obj.values():
            if isinstance(v, list) and v and isinstance(v[0], dict):
                return v
    return []

def public_path_label(p: Path) -> str:
    # Hide absolute HD roots in publishable prints.
    s = str(p)
    for key in ("DUMP_ROOT", "LAKE_ROOT", "COLLECT_ROOT"):
        root = os.getenv(key)
        if root and s.startswith(root):
            return f"${{{key}}}/" + s[len(root):].lstrip("/")
    if s.startswith(str(MONO)):
        return "mono:/" + s[len(str(MONO)):].lstrip("/")
    return p.name

print("extracao_ref", EXTRACAO, "| sample_dir", public_path_label(C.sample_dir()))
print("FIG_DIR", public_path_label(FIG_DIR))
print("SEED", SEED, "| CPU budget: light grids (SARIMAX small, HGB ≤6 combos, MLP early-stop)")


extracao_ref 2026-03 | sample_dir ${DUMP_ROOT}/extracao=2026-03/samples
FIG_DIR mono:/projects/monitoramento/output/figures
SEED 42 | CPU budget: light grids (SARIMAX small, HGB ≤6 combos, MLP early-stop)


### 2.1 Extração das séries e construção do painel

Preferência: Parquet local sob `samples/` da extração. Fallback: endpoints agregados `/v1/debito/estoque` e `/v1/arrecadacao/serie` (não *row-level*). Grão: mês-calendário; `data` = dia 1 do mês.


In [2]:
parquet_path = C.sample_dir() / "painel_mensal_estoque_arrecadacao.parquet"
panel = None
source = None
if parquet_path.exists():
    panel = pd.read_parquet(parquet_path)
    source = "parquet_local"
else:
    estoque = pd.DataFrame(first_list(C.get_json("/v1/debito/estoque")))
    arrec = pd.DataFrame(first_list(C.get_json("/v1/arrecadacao/serie")))
    for df_ in (estoque, arrec):
        df_.rename(columns={c: c.lower() for c in df_.columns}, inplace=True)
    panel = estoque.merge(arrec, on=["ano", "mes"], how="inner", suffixes=("_estoque", "_arrec"))
    source = "api_series"

panel = panel.sort_values(["ano", "mes"]).reset_index(drop=True)
panel.rename(columns={c: c.lower() for c in panel.columns}, inplace=True)
if "data" not in panel.columns:
    panel["data"] = pd.to_datetime(dict(year=panel["ano"], month=panel["mes"], day=1))
else:
    panel["data"] = pd.to_datetime(panel["data"])

qtd_candidates = [c for c in panel.columns if c.startswith("qtd") and "debito" not in c]
if "valor_total" in panel.columns and qtd_candidates and "ticket_medio" not in panel.columns:
    panel["ticket_medio"] = panel["valor_total"] / panel[qtd_candidates[0]].replace(0, np.nan)

panel.to_parquet(parquet_path, index=False)
print("source", source, "| painel", panel.shape, "->", public_path_label(parquet_path))
print("periodo", panel["data"].min().date(), "→", panel["data"].max().date())
panel.head(3)


source parquet_local | painel (119, 14) -> ${DUMP_ROOT}/extracao=2026-03/samples/painel_mensal_estoque_arrecadacao.parquet
periodo 2016-01-01 → 2026-03-01


,ano,mes,qtd_debitos,valor_sem_honorarios,valor_honorarios,ticket_medio,qtd_gares,valor_total,valor_receita,juros_mora,multa_mora,acrescimo_financeiro,honorarios_advocaticios,data
0,2016,1,6355185,3.075679e+11,5.537150e+10,856.935439,236581,2.027346e+08,1.298916e+08,31005572.76,16601086.33,19870333.25,5.366024e+06,2016-01-01
1,2016,2,6460669,3.100254e+11,5.560295e+10,941.315801,265594,2.500078e+08,1.611784e+08,39135377.56,23685748.04,20559247.73,5.449047e+06,2016-02-01
2,2016,3,5964885,3.105625e+11,5.589934e+10,1190.417525,303712,3.615441e+08,2.337149e+08,66306964.05,29132928.29,22383804.99,1.000545e+07,2016-03-01


### 2.2 Dicionário de variáveis (painel mensal)

| Variável | Origem | Descrição |
|---|---|---|
| `ano`, `mes` | ambos | Grão temporal; chave do merge. |
| `data` | derivado | Dia 1 do mês (eixos temporais). |
| `valor_sem_honorarios` | estoque | Estoque inscrito (R$) — *stock*. |
| `qtd_debitos` | estoque | Contagem de débitos (se presente). |
| `valor_total` | arrecadação | **Alvo $y$** do forecast — fluxo mensal (R$). |
| `qtd_gares` | arrecadação | Contagem de guias GARE (se presente). |
| `ticket_medio` | derivado | `valor_total / qtd` (descritivo). |


## 3. Visualização exploratória (Plotly interativo + export estático)

Cada figura é construída em **Plotly**: (1) `fig.show()` para exploração interativa (hover/zoom); (2) `fig.write_image(..., format="png")` via **Kaleido** para artefato estático de publicação. Encoding explícito; eixos rotulados; sem *chartjunk*.


### 3.1 Séries temporais: estoque versus arrecadação

**Eixos / encoding.** Eixo X = `data` (mês-calendário). Eixo Y **esquerdo** = estoque `valor_sem_honorarios` (R$). Eixo Y **direito** = arrecadação `valor_total` (R$). Duas traços (`Scatter`) com `secondary_y=True` porque estoque tipicamente $\gg$ fluxo mensal — um único eixo comprimiria a arrecadação.

**Comparação legítima.** Forma temporal e coincidência de picos/vales **dentro de cada eixo**.

**O que *não* afirmar.** Causalidade estoque→arrecadação; identidade contábil; igualdade de magnitudes entre eixos (caveat dual-axis, §1).


In [3]:
fig_dual = make_subplots(specs=[[{"secondary_y": True}]])
fig_dual.add_trace(
    go.Scatter(x=panel["data"], y=panel["valor_sem_honorarios"],
               name="Estoque (valor_sem_honorarios)", mode="lines",
               line=dict(width=1.8, color="#4C78A8")),
    secondary_y=False,
)
fig_dual.add_trace(
    go.Scatter(x=panel["data"], y=panel["valor_total"],
               name="Arrecadação (valor_total)", mode="lines",
               line=dict(width=1.8, color="#F58518")),
    secondary_y=True,
)
fig_dual.update_layout(
    title=dict(text="Estoque inscrito vs arrecadação mensal", y=0.98, x=0.5, xanchor="center"),
    height=480,
    margin=dict(t=90, l=70, r=70, b=50),
    legend=dict(orientation="h", yanchor="bottom", y=1.06, x=0, xanchor="left"),
    template="plotly_white",
)
fig_dual.update_xaxes(title_text="Mês")
fig_dual.update_yaxes(title_text="Estoque (R$)", secondary_y=False)
fig_dual.update_yaxes(title_text="Arrecadação (R$)", secondary_y=True)
fig_dual.show()
save_fig(fig_dual, "estoque_dual_axis", width=1100, height=480)


static -> mono:/projects/monitoramento/output/figures/estoque_dual_axis.png


PosixPath('/Users/etorebraga/Code/cemepi-ctf-intel-fiscal/projects/monitoramento/output/figures/estoque_dual_axis.png')

### 3.2 Perfil sazonal da arrecadação

**Eixos / encoding.** Barras: X = mês-do-ano $\{1,\ldots,12\}$; Y = média amostral de `valor_total` (R$) ao longo do painel.

**Comparação legítima.** Altura relativa entre meses motiva o baseline sazonal $\hat{y}_t=y_{t-12}$ (FPP3 §5.2).

**O que *não* afirmar.** Média incondicional ignora tendência e mudanças de regime; não prova sazonalidade estacionária nem efeito calendário causal.


In [4]:
seas = panel.groupby("mes", as_index=False)["valor_total"].mean()
fig_seas = px.bar(
    seas, x="mes", y="valor_total",
    title="Perfil sazonal médio da arrecadação (por mês-calendário)",
    labels={"valor_total": "Média valor_total (R$)", "mes": "Mês"},
    template="plotly_white",
)
fig_seas.update_layout(height=400, margin=dict(t=80, l=60, r=30, b=50),
                       title=dict(y=0.98))
fig_seas.update_xaxes(dtick=1)
fig_seas.show()
save_fig(fig_seas, "estoque_perfil_sazonal", width=900, height=400)


static -> mono:/projects/monitoramento/output/figures/estoque_perfil_sazonal.png


PosixPath('/Users/etorebraga/Code/cemepi-ctf-intel-fiscal/projects/monitoramento/output/figures/estoque_perfil_sazonal.png')

### 3.3 Dispersão estoque × arrecadação com outliers IQR

**Regra (Tukey, 1977).** Em cada margem, outlier se $x\notin[Q_1-1{,}5\cdot\mathrm{IQR},\,Q_3+1{,}5\cdot\mathrm{IQR}]$; flag se outlier em estoque **ou** arrecadação.

**Eixos / encoding.** Scatter: X = estoque (mesmo mês ou lag-1); Y = `valor_total`; cor = `{inlier, outlier_IQR}`; reta OLS só **descritiva**.

**Comparação legítima.** Associação linear frágil em níveis; localização temporal dos outliers (hover).

**O que *não* afirmar.** Outlier estatístico $\neq$ anomalia jurídica; correlação/OLS em níveis **não** identificam efeito causal do estoque sobre a arrecadação.


In [5]:
sc = panel[["data", "ano", "mes", "valor_sem_honorarios", "valor_total"]].copy()
sc["estoque_lag1"] = sc["valor_sem_honorarios"].shift(1)

def iqr_mask(s: pd.Series) -> pd.Series:
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return (s < lo) | (s > hi)

sc["outlier_iqr"] = iqr_mask(sc["valor_sem_honorarios"]) | iqr_mask(sc["valor_total"])
sc["flag"] = np.where(sc["outlier_iqr"], "outlier_IQR", "inlier")

fig_sc1 = px.scatter(
    sc, x="valor_sem_honorarios", y="valor_total", color="flag",
    hover_data=["data"],
    title="Dispersão: estoque vs arrecadação (mesmo mês) — outliers IQR",
    labels={"valor_sem_honorarios": "Estoque (R$)", "valor_total": "Arrecadação (R$)"},
    color_discrete_map={"inlier": "#4C78A8", "outlier_IQR": "#E45756"},
    trendline="ols",
    template="plotly_white",
)
fig_sc1.update_layout(height=460, margin=dict(t=80, l=70, r=30, b=50), title=dict(y=0.98))
fig_sc1.show()
save_fig(fig_sc1, "estoque_scatter_mesmo_mes", width=900, height=460)

fig_sc2 = px.scatter(
    sc.dropna(subset=["estoque_lag1"]),
    x="estoque_lag1", y="valor_total", color="flag",
    hover_data=["data"],
    title="Dispersão: estoque lag-1 vs arrecadação — outliers IQR",
    labels={"estoque_lag1": "Estoque t-1 (R$)", "valor_total": "Arrecadação t (R$)"},
    color_discrete_map={"inlier": "#4C78A8", "outlier_IQR": "#E45756"},
    trendline="ols",
    template="plotly_white",
)
fig_sc2.update_layout(height=460, margin=dict(t=80, l=70, r=30, b=50), title=dict(y=0.98))
fig_sc2.show()
save_fig(fig_sc2, "estoque_scatter_lag1", width=900, height=460)

corr_same = sc["valor_sem_honorarios"].corr(sc["valor_total"])
corr_lag = sc["estoque_lag1"].corr(sc["valor_total"])
print(f"corr(estoque, arrec) mesmo mês = {corr_same:.3f}")
print(f"corr(estoque_lag1, arrec) = {corr_lag:.3f}")
print(f"outliers_IQR = {int(sc['outlier_iqr'].sum())} / {len(sc)}")
print("Nota: validar outliers com domínio jurídico — não causalizar.")


static -> mono:/projects/monitoramento/output/figures/estoque_scatter_mesmo_mes.png


static -> mono:/projects/monitoramento/output/figures/estoque_scatter_lag1.png
corr(estoque, arrec) mesmo mês = 0.001
corr(estoque_lag1, arrec) = -0.010
outliers_IQR = 16 / 119
Nota: validar outliers com domínio jurídico — não causalizar.


### 3.4 Composição: ICMS (e IPVA se presente) em escala/facet separados

**Eixos / encoding.** Barras de estoque por tipo (`/v1/analytics/visao-geral`); facets **ICMS / IPVA / Demais** com eixos Y **independentes** (`matches=None`) porque as escalas diferem por ordens de magnitude. Segunda figura: arrecadação agregada por ano (contexto).

**O que *não* afirmar.** Mix de tipos **não** entra no forecast deste notebook; não decompor causalmente a série `valor_total`.


In [6]:
visao_ok = False
try:
    visao = C.get_json("/v1/analytics/visao-geral", timeout=45)
    tipo = pd.DataFrame(visao.get("estoque_por_tipo") or [])
    arrec_ano = pd.DataFrame(visao.get("arrecadacao_por_ano") or [])
    if not tipo.empty:
        tipo.columns = [c.lower() for c in tipo.columns]
        val_col = "valor" if "valor" in tipo.columns else tipo.select_dtypes("number").columns[0]
        tipo_col = "tipo" if "tipo" in tipo.columns else tipo.columns[0]
        tipo[val_col] = pd.to_numeric(tipo[val_col], errors="coerce")
        tnorm = tipo[tipo_col].astype(str).str.upper()
        tipo["grupo"] = np.where(
            tnorm.str.contains("IPVA"), "IPVA",
            np.where(tnorm.str.contains("ICMS"), "ICMS", "Demais"),
        )
        fig_tipo = px.bar(
            tipo.sort_values(val_col, ascending=False),
            x=tipo_col, y=val_col, facet_row="grupo",
            title="Estoque por tipo — ICMS / IPVA / Demais (escalas independentes por facet)",
            labels={val_col: "R$", tipo_col: "Tipo"},
            template="plotly_white",
        )
        fig_tipo.update_yaxes(matches=None)
        fig_tipo.update_layout(height=720, xaxis_tickangle=-35, margin=dict(t=80, l=60, r=30, b=80))
        fig_tipo.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
        fig_tipo.show()
        save_fig(fig_tipo, "estoque_composicao_tipo", width=1000, height=720)
        visao_ok = True
        print(tipo.groupby("grupo")[val_col].sum().sort_values(ascending=False))
    if not arrec_ano.empty:
        arrec_ano.columns = [c.lower() for c in arrec_ano.columns]
        ycol = "valor_total" if "valor_total" in arrec_ano.columns else arrec_ano.select_dtypes("number").columns[-1]
        xcol = "ano" if "ano" in arrec_ano.columns else arrec_ano.columns[0]
        fig_aa = px.bar(
            arrec_ano.sort_values(xcol), x=xcol, y=ycol,
            title="Arrecadação agregada por ano (visão-geral)",
            labels={ycol: "R$", xcol: "Ano"},
            template="plotly_white",
        )
        fig_aa.update_layout(height=380, margin=dict(t=80, l=60, r=30, b=50), title=dict(y=0.98))
        fig_aa.show()
        save_fig(fig_aa, "estoque_arrec_por_ano", width=900, height=380)
except Exception as e:
    print("visao-geral skip:", type(e).__name__, e)
if not visao_ok:
    print("Composição por tipo indisponível nesta execução — dual-axis e scatters permanecem.")


static -> mono:/projects/monitoramento/output/figures/estoque_composicao_tipo.png
grupo
ICMS      4.487653e+11
Demais    1.928179e+10
IPVA      7.708046e+09
Name: valor, dtype: float64


static -> mono:/projects/monitoramento/output/figures/estoque_arrec_por_ano.png


## 4. Protocolo de previsão (*walk-forward* / holdout $\approx$ 12m)

Métricas **MAPE / sMAPE / RMSE** — fórmulas e limitações na §1; aplicadas na **mesma** janela de holdout, $y=$ `valor_total`. Ranking primário por MAPE; sMAPE e RMSE como complementaridade. **Não** misturar AIC de treino (SARIMAX) com erro OOS (FPP3).

### Baselines e modelos (escada M0–M5)

| ID | Modelo | Âncora | Protocolo *aqui* |
|---|---|---|---|
| M0 | Naive lag-1 / sazonal lag-12 | FPP3 §5.2 | $\hat{y}_t=y_{t-1}$ ou $y_{t-12}$ |
| M1 | OLS / Ridge (lags 1,3,12 + estoque) | OLS clássico; Ridge (Hoerl & Kennard, 1970) | walk-forward 1-step; PI $\approx\pm 1{,}96\,\hat\sigma$ resíduo treino |
| M2 | SARIMAX sazonal | Box–Jenkins / FPP3 ARIMA | grid pequeno; AIC no treino; PI nativo |
| M3 | HistGradientBoosting | Friedman (2001) GBM; `sklearn` HGB | tune em val temporal; PI **empírico** |
| M4 | Prophet | Taylor & Letham (2018) | `yhat` + `yhat_lower/upper` nativos |
| M5 | MLP 2 camadas (PyTorch) | feed-forward rasa | janela de lags; early stop; PI empírico |

Alvo **$y$** = `valor_total`. Seed $=42$. Budget CPU: grid SARIMAX $\le\sim 8$ specs; HGB $\le 6$ combos; MLP $\le 200$ epochs com early stop.


In [7]:
y = panel["valor_total"].astype(float).copy()
n = len(y)
hold = 12 if n > 24 else max(3, n // 5)
train_end = n - hold
dates_h = panel["data"].iloc[train_end:n].reset_index(drop=True)
y_hold = y.iloc[train_end:n].values.astype(float)

def mape(a, f):
    a, f = np.asarray(a, float), np.asarray(f, float)
    m = a != 0
    return float(np.mean(np.abs((a[m] - f[m]) / a[m]))) if m.any() else np.nan

def smape(a, f):
    a, f = np.asarray(a, float), np.asarray(f, float)
    denom = np.abs(a) + np.abs(f)
    out = np.zeros_like(a, dtype=float)
    m = denom != 0
    out[m] = 2.0 * np.abs(a[m] - f[m]) / denom[m]
    return float(np.mean(out))

def rmse(a, f):
    return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(f, float)) ** 2)))

def score_row(name, actual, fcast, notes=""):
    return {
        "modelo": name,
        "MAPE": mape(actual, fcast),
        "SMAPE": smape(actual, fcast),
        "RMSE": rmse(actual, fcast),
        "notes": notes,
    }

results = []
preds = {}          # name -> np.array len hold (aligned to holdout; NaN if missing step)
intervals = {}      # name -> (lower, upper) arrays or None
residuals = {}      # name -> y - yhat
t0_all = time.time()

# ---------- M0 naive ----------
for name, lag in [("M0_naive_lag1", 1), ("M0_naive_sazonal_lag12", 12)]:
    fcast = np.array([float(y.iloc[t - lag]) if t - lag >= 0 else np.nan for t in range(train_end, n)], float)
    mask = ~np.isnan(fcast)
    results.append(score_row(name, y_hold[mask], fcast[mask], "baseline"))
    preds[name] = fcast
    residuals[name] = y_hold - fcast

print("M0 done")


M0 done


### 4.1 M1 — OLS / Ridge com lags 1, 3, 12 (+ estoque lag)

**OLS.** Coeficientes $\hat\beta=\arg\min_\beta\|y-X\beta\|_2^2$.

**Ridge** (Hoerl & Kennard, 1970):

$$
\hat\beta_{\mathrm{ridge}}=\arg\min_\beta\bigl(\|y-X\beta\|_2^2+\alpha\|\beta\|_2^2\bigr),
\quad \alpha=1 \text{ neste v0}.
$$

**Aplicação aqui.** Features: arrecadação e estoque em lags $1,3,12$. *Walk-forward* 1-passo (FPP3 §5.10): a cada $t$ do holdout, refit só com $t'<t$.

**PI.** $\hat{y}\pm 1{,}96\,\hat\sigma$ dos resíduos de treino — homocedasticidade **não** garantida.

**Limitações.** Linearidade; colinearidade entre lags; PI clássico aproximado, não bayesiano.


In [8]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler

feat = panel[["valor_total", "valor_sem_honorarios"]].astype(float).copy()
for lag in (1, 3, 12):
    feat[f"arrec_l{lag}"] = feat["valor_total"].shift(lag)
    feat[f"estoque_l{lag}"] = feat["valor_sem_honorarios"].shift(lag)
feat = feat.dropna()
Xcols = [c for c in feat.columns if c.startswith("arrec_l") or c.startswith("estoque_l")]

def walk_forward_tabular(model_factory, name, alpha_note=""):
    fcast_list, actual_list, idx_list, resid_train_last = [], [], [], None
    lowers, uppers = [], []
    for t in range(train_end, n):
        if t not in feat.index:
            fcast_list.append(np.nan); actual_list.append(y.iloc[t]); idx_list.append(t)
            lowers.append(np.nan); uppers.append(np.nan)
            continue
        tr = feat.loc[feat.index < t]
        if len(tr) < 24:
            fcast_list.append(np.nan); actual_list.append(float(feat.loc[t, "valor_total"]))
            idx_list.append(t); lowers.append(np.nan); uppers.append(np.nan)
            continue
        mdl = model_factory()
        Xtr, ytr = tr[Xcols].values, tr["valor_total"].values
        mdl.fit(Xtr, ytr)
        yhat = float(mdl.predict(feat.loc[[t], Xcols].values)[0])
        rtr = ytr - mdl.predict(Xtr)
        resid_train_last = rtr
        sigma = float(np.std(rtr, ddof=1)) if len(rtr) > 2 else np.nan
        fcast_list.append(yhat)
        actual_list.append(float(feat.loc[t, "valor_total"]))
        idx_list.append(t)
        lowers.append(yhat - 1.96 * sigma if np.isfinite(sigma) else np.nan)
        uppers.append(yhat + 1.96 * sigma if np.isfinite(sigma) else np.nan)
    fcast = np.array(fcast_list, float)
    actual = np.array(actual_list, float)
    mask = ~np.isnan(fcast)
    results.append(score_row(name, actual[mask], fcast[mask], alpha_note))
    preds[name] = fcast
    residuals[name] = actual - fcast
    intervals[name] = (np.array(lowers, float), np.array(uppers, float))
    return actual, fcast

walk_forward_tabular(lambda: LinearRegression(), "M1_OLS_lags", "PI≈±1.96σ resíduo treino")
walk_forward_tabular(lambda: Ridge(alpha=1.0, random_state=SEED), "M1_Ridge_lags", "Ridge α=1; PI≈±1.96σ")
print("M1 done")


M1 done


### 4.2 M2 — SARIMAX com grid pequeno

**O que é.** Extensão sazonal do ARIMA (tradição Box–Jenkins; FPP3, capítulos ARIMA/SARIMA). Ordem $(p,d,q)\times(P,D,Q)_{s=12}$.

**Aplicação aqui.** Grid restrito; seleção por **AIC no treino**; reporte MAPE/sMAPE/RMSE no holdout. Lembrete FPP3: AIC $\neq$ erro OOS.

**PI.** Nativo via `get_forecast(...).summary_frame(alpha=0.05)`.

**Limitações.** One-shot multi-passo (não re-fit recursivo completo); grid pequeno pode omitir ordem ótima OOS.


In [9]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarimax_grid = [
    ((1, 1, 0), (0, 1, 1, 12)),
    ((1, 1, 1), (0, 1, 1, 12)),
    ((1, 1, 1), (1, 0, 0, 12)),
    ((0, 1, 1), (0, 1, 1, 12)),
    ((2, 1, 0), (1, 0, 0, 12)),
    ((1, 1, 0), (1, 0, 0, 12)),
]
best_aic, best_spec, best_fit = np.inf, None, None
y_tr = y.iloc[:train_end]
if train_end >= 36:
    for order, seasonal in sarimax_grid:
        try:
            fit = SARIMAX(
                y_tr, order=order, seasonal_order=seasonal,
                enforce_stationarity=False, enforce_invertibility=False,
            ).fit(disp=False, maxiter=100)
            aic = float(fit.aic)
            tag = f"SARIMAX{order}x{seasonal}"
            print(f"  grid {tag} AIC={aic:.1f}")
            if aic < best_aic:
                best_aic, best_spec, best_fit = aic, (order, seasonal), fit
        except Exception as e:
            print("  skip", order, seasonal, type(e).__name__)
    if best_fit is not None:
        order, seasonal = best_spec
        fc = best_fit.get_forecast(hold)
        sf = fc.summary_frame(alpha=0.05)
        # columns typically: mean, mean_se, mean_ci_lower, mean_ci_upper
        mean_col = "mean" if "mean" in sf.columns else sf.columns[0]
        lo_col = [c for c in sf.columns if "lower" in c.lower()][0]
        hi_col = [c for c in sf.columns if "upper" in c.lower()][0]
        fcast = sf[mean_col].values.astype(float)
        lo = sf[lo_col].values.astype(float)
        hi = sf[hi_col].values.astype(float)
        name = f"M2_SARIMAX{order}x{seasonal}"
        results.append(score_row(name, y_hold, fcast, f"AIC_train={best_aic:.1f}; PI nativo 95%"))
        preds[name] = fcast
        residuals[name] = y_hold - fcast
        intervals[name] = (lo, hi)
        print("M2 best", name, "AIC", best_aic)
    else:
        print("M2: nenhum fit ok")
else:
    print("M2: série curta — skip")


  grid SARIMAX(1, 1, 0)x(0, 1, 1, 12) AIC=3375.3
  grid SARIMAX(1, 1, 1)x(0, 1, 1, 12) AIC=3323.7
  grid SARIMAX(1, 1, 1)x(1, 0, 0, 12) AIC=3838.3
  grid SARIMAX(0, 1, 1)x(0, 1, 1, 12) AIC=3322.6
  grid SARIMAX(2, 1, 0)x(1, 0, 0, 12) AIC=3816.4
  grid SARIMAX(1, 1, 0)x(1, 0, 0, 12) AIC=3863.4


M2 best M2_SARIMAX(0, 1, 1)x(0, 1, 1, 12) AIC 3322.621771758574


### 4.3 M3 — HistGradientBoosting tunado

**O que é.** Ensemble por *gradient boosting* (Friedman, 2001); implementação com histogramas: `sklearn.ensemble.HistGradientBoostingRegressor` (documentação scikit-learn).

**Aplicação aqui.** Busca leve: `max_iter` $\in\{200,500\}$, `learning_rate` $\in\{0.05,0.1\}$, `max_depth` $\in\{3,5\}$; escolha em fatia temporal final do treino; depois walk-forward no holdout.

**PI empírico.** $\hat{y}\pm$ quantis 5/95 dos resíduos OOS — **não** intervalo bayesiano; cobertura nominal **não** garantida (FPP3 §5.5).

**Limitações.** Hiperparâmetros e regime; PI empírico não calibrado.


In [10]:
from sklearn.ensemble import HistGradientBoostingRegressor

hgb_grid = [
    {"max_iter": 200, "learning_rate": 0.05, "max_depth": 3},
    {"max_iter": 200, "learning_rate": 0.1, "max_depth": 3},
    {"max_iter": 500, "learning_rate": 0.05, "max_depth": 3},
    {"max_iter": 500, "learning_rate": 0.1, "max_depth": 3},
    {"max_iter": 200, "learning_rate": 0.05, "max_depth": 5},
    {"max_iter": 500, "learning_rate": 0.1, "max_depth": 5},
]
# temporal validation slice inside training
val_h = min(12, max(3, train_end // 5))
val_start = train_end - val_h
best_hgb, best_hgb_mape, best_hgb_params = None, np.inf, None
for params in hgb_grid:
    fcs, acts = [], []
    for t in range(val_start, train_end):
        if t not in feat.index:
            continue
        tr = feat.loc[feat.index < t]
        if len(tr) < 24:
            continue
        mdl = HistGradientBoostingRegressor(random_state=SEED, early_stopping=True, validation_fraction=0.15, **params)
        mdl.fit(tr[Xcols], tr["valor_total"])
        fcs.append(float(mdl.predict(feat.loc[[t], Xcols])[0]))
        acts.append(float(feat.loc[t, "valor_total"]))
    if fcs:
        m = mape(acts, fcs)
        print(f"  HGB val MAPE={m:.4f} params={params}")
        if m < best_hgb_mape:
            best_hgb_mape, best_hgb_params = m, params

print("M3 best params", best_hgb_params, "val_MAPE", best_hgb_mape)
fcast_list, actual_list, oos_resid = [], [], []
if best_hgb_params:
    for t in range(train_end, n):
        if t not in feat.index:
            fcast_list.append(np.nan); actual_list.append(float(y.iloc[t])); continue
        tr = feat.loc[feat.index < t]
        if len(tr) < 24:
            fcast_list.append(np.nan); actual_list.append(float(feat.loc[t, "valor_total"])); continue
        mdl = HistGradientBoostingRegressor(random_state=SEED, early_stopping=True, validation_fraction=0.15, **best_hgb_params)
        mdl.fit(tr[Xcols], tr["valor_total"])
        yhat = float(mdl.predict(feat.loc[[t], Xcols])[0])
        yt = float(feat.loc[t, "valor_total"])
        fcast_list.append(yhat); actual_list.append(yt); oos_resid.append(yt - yhat)
    fcast = np.array(fcast_list, float)
    actual = np.array(actual_list, float)
    mask = ~np.isnan(fcast)
    name = "M3_HGB_tuned"
    results.append(score_row(name, actual[mask], fcast[mask], f"params={best_hgb_params}; PI empírico quantil resíduo OOS"))
    preds[name] = fcast
    residuals[name] = actual - fcast
    # empirical PI from OOS residual quantiles (honest label)
    if len(oos_resid) >= 4:
        q5, q95 = np.quantile(oos_resid, [0.05, 0.95])
        intervals[name] = (fcast + q5, fcast + q95)
    print("M3 done")
else:
    print("M3 skip — sem params")


  HGB val MAPE=0.3079 params={'max_iter': 200, 'learning_rate': 0.05, 'max_depth': 3}


  HGB val MAPE=0.3006 params={'max_iter': 200, 'learning_rate': 0.1, 'max_depth': 3}


  HGB val MAPE=0.3079 params={'max_iter': 500, 'learning_rate': 0.05, 'max_depth': 3}


  HGB val MAPE=0.3006 params={'max_iter': 500, 'learning_rate': 0.1, 'max_depth': 3}


  HGB val MAPE=0.3079 params={'max_iter': 200, 'learning_rate': 0.05, 'max_depth': 5}


  HGB val MAPE=0.3006 params={'max_iter': 500, 'learning_rate': 0.1, 'max_depth': 5}
M3 best params {'max_iter': 200, 'learning_rate': 0.1, 'max_depth': 3} val_MAPE 0.3005531836850197


M3 done


### 4.4 M4 — Prophet

**O que é.** Decomposição aditiva tendência + sazonalidade + feriados (Taylor & Letham, 2018, *The American Statistician*; preprint PeerJ Preprints / Facebook Prophet).

**Aplicação aqui.** Fit no treino; `predict` no horizonte do holdout; sazonalidade anual; sem feriados customizados neste v0.

**PI.** Nativo `yhat_lower` / `yhat_upper` (`interval_width=0.95`).

**Limitações.** Sensível a mudanças de regime e a defaults; ranking condicional a este holdout.


In [11]:
try:
    from prophet import Prophet
    df_p = pd.DataFrame({
        "ds": panel["data"].iloc[:train_end].values,
        "y": y.iloc[:train_end].values,
    })
    m_prophet = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        interval_width=0.95,
    )
    m_prophet.fit(df_p)
    future = pd.DataFrame({"ds": panel["data"].iloc[train_end:n].values})
    fc_p = m_prophet.predict(future)
    fcast = fc_p["yhat"].values.astype(float)
    lo = fc_p["yhat_lower"].values.astype(float)
    hi = fc_p["yhat_upper"].values.astype(float)
    name = "M4_Prophet"
    results.append(score_row(name, y_hold, fcast, "PI nativo yhat_lower/upper"))
    preds[name] = fcast
    residuals[name] = y_hold - fcast
    intervals[name] = (lo, hi)
    print("M4 done")
except Exception as e:
    print("M4 Prophet skip:", type(e).__name__, e)


12:42:07 - cmdstanpy - INFO - Chain [1] start processing


12:42:08 - cmdstanpy - INFO - Chain [1] done processing


M4 done


### 4.5 M5 — MLP 2 camadas (PyTorch)

**O que é.** Perceptron multicamadas rasa: $\mathrm{Linear}\to\mathrm{ReLU}\to\mathrm{Linear}\to\mathrm{ReLU}\to\mathrm{Linear}$ — aproximador não-linear de janela de lags.

**Aplicação aqui.** Entrada = 12 lags de arrecadação + 12 de estoque; standardização só no treino; early stopping em fatia temporal de validação ($\le 200$ epochs).

**PI empírico.** Quantis de resíduos OOS (rótulo: empírico).

**Limitações.** *Smoke* de DL mínimo — gate para DL profundo permanece fechado se baselines vencerem (FPP3).


In [12]:
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset

    torch.manual_seed(SEED)
    LAG_WINDOW = 12
    # build supervised matrix from full panel
    arr = y.values.astype(np.float64)
    est = panel["valor_sem_honorarios"].astype(float).values
    Xs, ys_t, idxs = [], [], []
    for t in range(LAG_WINDOW, n):
        x = np.concatenate([arr[t-LAG_WINDOW:t], est[t-LAG_WINDOW:t]])
        Xs.append(x); ys_t.append(arr[t]); idxs.append(t)
    Xs = np.asarray(Xs, np.float32)
    ys_t = np.asarray(ys_t, np.float32)
    idxs = np.asarray(idxs)

    # standardize using training portion only
    train_mask_idx = idxs < train_end
    mu = Xs[train_mask_idx].mean(axis=0)
    sd = Xs[train_mask_idx].std(axis=0) + 1e-8
    ymu = ys_t[train_mask_idx].mean()
    ysd = ys_t[train_mask_idx].std() + 1e-8
    Xn = (Xs - mu) / sd
    yn = (ys_t - ymu) / ysd

    class MLP2(nn.Module):
        def __init__(self, d):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(d, 32), nn.ReLU(),
                nn.Linear(32, 16), nn.ReLU(),
                nn.Linear(16, 1),
            )
        def forward(self, x):
            return self.net(x).squeeze(-1)

    # fit once on train (with early stop on last val_h of train idxs)
    tr_idx = np.where(idxs < train_end)[0]
    if len(tr_idx) > val_h + 10:
        va_local = tr_idx[-val_h:]
        tr_local = tr_idx[:-val_h]
    else:
        tr_local, va_local = tr_idx, tr_idx[-max(3, len(tr_idx)//5):]

    device = torch.device("cpu")
    model = MLP2(Xn.shape[1]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    best_state, best_va, patience, bad = None, np.inf, 15, 0
    Xtr = torch.tensor(Xn[tr_local], device=device)
    ytr = torch.tensor(yn[tr_local], device=device)
    Xva = torch.tensor(Xn[va_local], device=device)
    yva = torch.tensor(yn[va_local], device=device)
    for epoch in range(200):
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(Xtr), ytr)
        loss.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            va_loss = float(loss_fn(model(Xva), yva).item())
        if va_loss < best_va - 1e-5:
            best_va = va_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break
    if best_state:
        model.load_state_dict(best_state)
    model.eval()

    # predictions on holdout indices present in idxs
    fcast = np.full(hold, np.nan)
    oos_resid = []
    with torch.no_grad():
        for i, t in enumerate(range(train_end, n)):
            hit = np.where(idxs == t)[0]
            if len(hit) == 0:
                continue
            pred_n = float(model(torch.tensor(Xn[hit], device=device)).cpu().numpy()[0])
            yhat = pred_n * ysd + ymu
            fcast[i] = yhat
            oos_resid.append(y_hold[i] - yhat)
    mask = ~np.isnan(fcast)
    name = "M5_MLP2_torch"
    results.append(score_row(name, y_hold[mask], fcast[mask], f"lag_window={LAG_WINDOW}; early_stop; epochs≤200; PI empírico"))
    preds[name] = fcast
    residuals[name] = y_hold - fcast
    if len(oos_resid) >= 4:
        q5, q95 = np.quantile(oos_resid, [0.05, 0.95])
        intervals[name] = (fcast + q5, fcast + q95)
    print("M5 done; best_va", best_va, "epochs_stopped_patience", bad)
except Exception as e:
    print("M5 MLP skip:", type(e).__name__, e)


M5 done; best_va 2.224710464477539 epochs_stopped_patience 15


### 4.6 Tabela e figuras de comparação no holdout (todos os modelos)

**Tabela.** Uma linha por modelo M0–M5 avaliado; mesmas janela e $y$. Menor MAPE/sMAPE/RMSE = melhor *neste* holdout (não generalização universal).

**Figura A — barras de métricas (todos os modelos).** Facet horizontal por métrica (MAPE | sMAPE | RMSE); modelos no eixo Y, **ordenados por sMAPE ascendente**; RMSE em escala própria (facet). Barras mais curtas = melhor naquela métrica.

**Figura B — série holdout com todos os modelos.** Eixo X = mês do holdout; Y = `valor_total` (R$). Traço grosso = **atual**; demais = $\hat{y}$ de **cada** modelo avaliado (cores/linestyles distintas, legenda horizontal). Permite comparar trajetórias, não só o escalar de erro.

**O que *não* afirmar.** Superioridade metodológica definitiva; validade fora de `extracao=2026-03`; que AIC (SARIMAX) preveja o ranking OOS; que sobreposição visual implique equivalência estatística.


In [13]:
res_df = pd.DataFrame(results).sort_values("MAPE").reset_index(drop=True)
print("Holdout meses:", hold, "| train_end:", train_end, "| n:", n, "| elapsed_s:", round(time.time() - t0_all, 1))
try:
    display(res_df)
except NameError:
    from IPython.display import display
    display(res_df)
best_name = res_df.iloc[0]["modelo"] if len(res_df) else None
if best_name is not None:
    print(
        f"BEST_MODEL={best_name} "
        f"MAPE={res_df.iloc[0]['MAPE']:.6f} "
        f"SMAPE={res_df.iloc[0]['SMAPE']:.6f} "
        f"RMSE={res_df.iloc[0]['RMSE']:.2f}"
    )
metrics_path = MONO / ".local/scratch/monitoramento_faseA_metrics.json"
metrics_path.parent.mkdir(parents=True, exist_ok=True)
res_df.to_json(metrics_path, orient="records", indent=2)
print("metrics ->", public_path_label(metrics_path))

SHORT = {
    "M0_naive_lag1": "Naive lag-1",
    "M0_naive_sazonal_lag12": "Naive saz. lag-12",
    "M1_OLS_lags": "OLS lags",
    "M1_Ridge_lags": "Ridge lags",
    "M3_HGB_tuned": "HGB",
    "M4_Prophet": "Prophet",
    "M5_MLP2_torch": "MLP-2",
}

def short_name(m: str) -> str:
    if m in SHORT:
        return SHORT[m]
    if m.startswith("M2_SARIMAX"):
        return "SARIMAX"
    return m.replace("M0_", "").replace("M1_", "").replace("M3_", "").replace("M4_", "").replace("M5_", "")[:22]

def aligned_pred(name):
    p = preds.get(name)
    if p is None:
        return None
    p = np.asarray(p, float)
    if len(p) != hold:
        out = np.full(hold, np.nan)
        out[: min(hold, len(p))] = p[: min(hold, len(p))]
        return out
    return p

plot_df = res_df.copy()
plot_df["label"] = plot_df["modelo"].map(short_name)
order = plot_df.sort_values("SMAPE", ascending=True)["label"].tolist()
long = plot_df.melt(
    id_vars=["label", "modelo"],
    value_vars=["MAPE", "SMAPE", "RMSE"],
    var_name="metric", value_name="value",
)
long["label"] = pd.Categorical(long["label"], categories=order, ordered=True)
long["metric"] = pd.Categorical(long["metric"], categories=["MAPE", "SMAPE", "RMSE"], ordered=True)

# Figura A — all models metrics
fig_cmp = px.bar(
    long.sort_values(["metric", "label"]),
    x="value", y="label", facet_col="metric", facet_col_wrap=3,
    orientation="h",
    title="Holdout ~12m — TODOS os modelos (ordenado por sMAPE ↑)",
    labels={"value": "", "label": ""},
    category_orders={"label": order, "metric": ["MAPE", "SMAPE", "RMSE"]},
    template="plotly_white",
)
fig_cmp.update_xaxes(matches=None)
fig_cmp.for_each_xaxis(lambda ax: ax.update(showticklabels=True))
fig_cmp.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_cmp.update_layout(
    height=max(380, 48 * len(order) + 120),
    margin=dict(t=90, l=140, r=40, b=50),
    title=dict(y=0.98, x=0.5, xanchor="center"),
    showlegend=False,
)
fig_cmp.show()
save_fig(fig_cmp, "estoque_holdout_metrics_all_models", width=1100, height=max(380, 48 * len(order) + 120))

# Figura B — all models holdout time series
PALETTE = [
    "#E45756", "#4C78A8", "#72B7B2", "#F58518", "#54A24B",
    "#EECA3B", "#B279A2", "#FF9DA6", "#9D755D", "#BAB0AC",
]
DASHES = ["solid", "dash", "dot", "dashdot", "solid", "dash", "dot", "dashdot"]
fig_all = go.Figure()
fig_all.add_trace(go.Scatter(
    x=dates_h, y=y_hold, name="Atual", mode="lines+markers",
    line=dict(width=3, color="#222"), marker=dict(size=7),
))
for i, row in enumerate(res_df.itertuples(index=False)):
    nm = row.modelo
    pred = aligned_pred(nm)
    if pred is None or np.all(np.isnan(pred)):
        continue
    lab = short_name(nm)
    fig_all.add_trace(go.Scatter(
        x=dates_h, y=pred, name=lab, mode="lines+markers",
        line=dict(width=1.4, color=PALETTE[i % len(PALETTE)], dash=DASHES[i % len(DASHES)]),
        marker=dict(size=5), opacity=0.85,
    ))
fig_all.update_layout(
    title=dict(text=f"Holdout ({hold}m): atual vs TODOS os modelos — y=valor_total", y=0.98),
    height=560,
    margin=dict(t=110, l=70, r=40, b=50),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0, font=dict(size=11)),
    yaxis_title="Arrecadação valor_total (R$)",
    xaxis_title="Mês (holdout)",
    template="plotly_white",
)
fig_all.show()
save_fig(fig_all, "estoque_holdout_series_all_models", width=1200, height=560)


Holdout meses: 12 | train_end: 107 | n: 119 | elapsed_s: 64.1


,modelo,MAPE,SMAPE,RMSE,notes
0,M4_Prophet,0.110295,0.115441,1.005494e+08,PI nativo yhat_lower/upper
1,M3_HGB_tuned,0.122995,0.133351,1.124517e+08,"params={'max_iter': 200, 'learning_rate': 0.1,..."
2,M0_naive_sazonal_lag12,0.153495,0.135399,1.315793e+08,baseline
3,M0_naive_lag1,0.159695,0.160860,1.315259e+08,baseline
4,M1_Ridge_lags,0.202830,0.232595,1.485746e+08,Ridge α=1; PI≈±1.96σ
5,M1_OLS_lags,0.202830,0.232595,1.485746e+08,PI≈±1.96σ resíduo treino
6,"M2_SARIMAX(0, 1, 1)x(0, 1, 1, 12)",0.259859,0.226964,1.520269e+08,AIC_train=3322.6; PI nativo 95%
7,M5_MLP2_torch,0.390606,0.497594,2.640788e+08,lag_window=12; early_stop; epochs≤200; PI empí...


BEST_MODEL=M4_Prophet MAPE=0.110295 SMAPE=0.115441 RMSE=100549443.58
metrics -> mono:/.local/scratch/monitoramento_faseA_metrics.json


static -> mono:/projects/monitoramento/output/figures/estoque_holdout_metrics_all_models.png


static -> mono:/projects/monitoramento/output/figures/estoque_holdout_series_all_models.png


PosixPath('/Users/etorebraga/Code/cemepi-ctf-intel-fiscal/projects/monitoramento/output/figures/estoque_holdout_series_all_models.png')

### 4.7 Diagnóstico: real × previsto, resíduos e intervalos

**Figura C — real × previsto (melhor MAPE).** Pontos $(\,y_t,\hat{y}_t\,)$ no holdout; diagonal $y=\hat{y}$. Desvios sistemáticos acima/abaixo = viés de nível.

**Figura D — resíduos no tempo (melhor + top-3 por sMAPE).** $e_t=y_t-\hat{y}_t$; idealmente ruído sem tendência. Com $h\approx 12$, ACF é só descritivo.

**Figura E — holdout: atual + melhor + sazonal (+ banda PI).** Complementa a Figura B (todos os modelos) com foco no vencedor e no baseline sazonal; banda = PI nativo ou empírico (rótulo explícito).

**O que *não* afirmar.** Cobertura nominal 95% dos PI empíricos; ausência de autocorrelação residual com $n\approx 12$.


In [14]:
best_pred = aligned_pred(best_name) if best_name else None
if best_pred is None or np.all(np.isnan(best_pred)):
    for nm in res_df["modelo"]:
        best_pred = aligned_pred(nm)
        if best_pred is not None and not np.all(np.isnan(best_pred)):
            best_name = nm
            break

best_label = short_name(best_name) if best_name else "?"

# Figura C — scatter real vs pred (best)
fig_scatter = go.Figure()
fig_scatter.add_trace(go.Scatter(
    x=y_hold, y=best_pred, mode="markers", name=best_label,
    text=[str(d.date()) for d in dates_h],
    marker=dict(size=9, color="#4C78A8"),
))
lo_xy = float(np.nanmin([np.nanmin(y_hold), np.nanmin(best_pred)]))
hi_xy = float(np.nanmax([np.nanmax(y_hold), np.nanmax(best_pred)]))
fig_scatter.add_trace(go.Scatter(
    x=[lo_xy, hi_xy], y=[lo_xy, hi_xy], mode="lines", name="y = ŷ",
    line=dict(dash="dash", color="gray"),
))
fig_scatter.update_layout(
    title=dict(text=f"Holdout: real × previsto ({best_label})", y=0.98),
    xaxis_title="Real valor_total (R$)",
    yaxis_title="Previsto (R$)",
    height=460,
    margin=dict(t=80, l=70, r=40, b=50),
    legend=dict(orientation="h", yanchor="bottom", y=1.08, x=0),
    template="plotly_white",
)
fig_scatter.show()
save_fig(fig_scatter, "estoque_real_vs_pred_best", width=900, height=460)

# optional small multiples: real vs pred for each model
n_models = len(res_df)
n_cols = 2
n_rows = int(np.ceil(n_models / n_cols))
fig_sm = make_subplots(rows=n_rows, cols=n_cols,
                       subplot_titles=[short_name(m) for m in res_df["modelo"]],
                       horizontal_spacing=0.08, vertical_spacing=0.12)
for i, nm in enumerate(res_df["modelo"]):
    r, c = divmod(i, n_cols)
    pred = aligned_pred(nm)
    if pred is None:
        continue
    fig_sm.add_trace(
        go.Scatter(x=y_hold, y=pred, mode="markers",
                   marker=dict(size=6, color=PALETTE[i % len(PALETTE)]),
                   showlegend=False, name=short_name(nm)),
        row=r + 1, col=c + 1,
    )
    fig_sm.add_trace(
        go.Scatter(x=[lo_xy, hi_xy], y=[lo_xy, hi_xy], mode="lines",
                   line=dict(dash="dash", color="gray", width=1),
                   showlegend=False),
        row=r + 1, col=c + 1,
    )
fig_sm.update_layout(
    title=dict(text="Holdout: real × previsto — painéis por modelo", y=0.99),
    height=220 * n_rows + 80,
    margin=dict(t=80, l=60, r=30, b=40),
    template="plotly_white",
)
fig_sm.update_xaxes(title_text="Real (R$)")
fig_sm.update_yaxes(title_text="Previsto (R$)")
fig_sm.show()
save_fig(fig_sm, "estoque_real_vs_pred_all_panels", width=1000, height=220 * n_rows + 80)

# Figura D — residuals best + top-3 by SMAPE
top3 = res_df.sort_values("SMAPE").head(3)["modelo"].tolist()
fig_res = make_subplots(rows=len(top3), cols=1, shared_xaxes=True,
                        subplot_titles=[f"Resíduos: {short_name(m)}" for m in top3],
                        vertical_spacing=0.08)
for i, nm in enumerate(top3):
    pred = aligned_pred(nm)
    resid = y_hold - pred
    fig_res.add_trace(
        go.Scatter(x=dates_h, y=resid, mode="lines+markers",
                   name=short_name(nm),
                   line=dict(color=PALETTE[i % len(PALETTE)]),
                   showlegend=False),
        row=i + 1, col=1,
    )
    fig_res.add_hline(y=0, line_dash="dot", line_color="gray", row=i + 1, col=1)
fig_res.update_layout(
    title=dict(text="Resíduos no tempo (y − ŷ) — top-3 por sMAPE", y=0.98),
    height=200 * len(top3) + 80,
    margin=dict(t=80, l=70, r=40, b=50),
    template="plotly_white",
)
fig_res.update_yaxes(title_text="Resíduo (R$)")
fig_res.show()
save_fig(fig_res, "estoque_residuos_top3", width=1000, height=200 * len(top3) + 80)

try:
    from statsmodels.tsa.stattools import acf
    resid_best = y_hold - best_pred
    r = resid_best[np.isfinite(resid_best)]
    if len(r) >= 8:
        ac = acf(r, nlags=min(6, len(r) // 2), fft=True)
        fig_acf = px.bar(x=list(range(len(ac))), y=ac, labels={"x": "lag", "y": "ACF"},
                         title=f"ACF residual curto — {best_label}",
                         template="plotly_white")
        fig_acf.update_layout(height=340, margin=dict(t=80, l=60, r=30, b=40), title=dict(y=0.98))
        fig_acf.show()
        save_fig(fig_acf, "estoque_acf_best", width=800, height=340)
except Exception as e:
    print("ACF skip", type(e).__name__, e)

# Figura E — atual + best + seasonal (+ PI)
fig_fc = go.Figure()
fig_fc.add_trace(go.Scatter(x=dates_h, y=y_hold, name="Atual", mode="lines+markers",
                            line=dict(width=2.5, color="#333")))
seas_name = "M0_naive_sazonal_lag12"
if seas_name in preds:
    fig_fc.add_trace(go.Scatter(
        x=dates_h, y=aligned_pred(seas_name), name="Naive saz. lag-12",
        mode="lines+markers", line=dict(dash="dot", color="#72B7B2"),
    ))
fig_fc.add_trace(go.Scatter(
    x=dates_h, y=best_pred, name=f"Melhor: {best_label}",
    mode="lines+markers", line=dict(width=2, color="#E45756"),
))
if best_name in intervals:
    lo, hi = intervals[best_name]
    lo = np.asarray(lo, float); hi = np.asarray(hi, float)
    pi_tag = "PI nativo" if best_name.startswith("M2_") or best_name.startswith("M4_") else "PI empírico"
    fig_fc.add_trace(go.Scatter(
        x=list(dates_h) + list(dates_h[::-1]),
        y=list(hi) + list(lo[::-1]),
        fill="toself", fillcolor="rgba(228,87,86,0.18)",
        line=dict(color="rgba(255,255,255,0)"),
        name=f"{pi_tag} {best_label}",
        hoverinfo="skip",
    ))
fig_fc.update_layout(
    title=dict(text=f"Holdout ({hold}m): atual vs melhor + sazonal — {best_label}", y=0.98),
    height=520,
    margin=dict(t=100, l=70, r=40, b=50),
    legend=dict(orientation="h", yanchor="bottom", y=1.08, x=0),
    yaxis_title="Arrecadação valor_total (R$)",
    template="plotly_white",
)
fig_fc.show()
save_fig(fig_fc, "estoque_holdout_best_seasonal_pi", width=1100, height=520)

print(res_df.to_string(index=False))
print("FIG_DIR figures:", sorted(p.name for p in FIG_DIR.glob('estoque_*.png')))


static -> mono:/projects/monitoramento/output/figures/estoque_real_vs_pred_best.png


static -> mono:/projects/monitoramento/output/figures/estoque_real_vs_pred_all_panels.png


static -> mono:/projects/monitoramento/output/figures/estoque_residuos_top3.png


static -> mono:/projects/monitoramento/output/figures/estoque_acf_best.png


static -> mono:/projects/monitoramento/output/figures/estoque_holdout_best_seasonal_pi.png
                           modelo     MAPE    SMAPE         RMSE                                                                                           notes
                       M4_Prophet 0.110295 0.115441 1.005494e+08                                                                      PI nativo yhat_lower/upper
                     M3_HGB_tuned 0.122995 0.133351 1.124517e+08 params={'max_iter': 200, 'learning_rate': 0.1, 'max_depth': 3}; PI empírico quantil resíduo OOS
           M0_naive_sazonal_lag12 0.153495 0.135399 1.315793e+08                                                                                        baseline
                    M0_naive_lag1 0.159695 0.160860 1.315259e+08                                                                                        baseline
                    M1_Ridge_lags 0.202830 0.232595 1.485746e+08                                        

## 5. Limitações

- **Dump estático:** resultados condicionados a `extracao=2026-03`.
- **Tipagem frágil** em outros endpoints; aqui usamos agregados tipados.
- **Cobertura:** sem garantia nem dívida **não inscrita**.
- **API row-level:** timeouts motivam agregados / Parquet local.
- **SARIMAX one-shot** multi-passo (não re-fit recursivo completo).
- **PI empíricos (HGB/MLP):** quantis de resíduos OOS — **não** são intervalos bayesianos nem garantem cobertura nominal.
- **Prophet / MLP:** sensíveis a hiperparâmetros e a mudanças de regime; ranking é condicional a este holdout.
- **Outliers IQR:** regra estatística; validação de domínio jurídico pendente — não causalizar.


## 6. Conclusões e próximos passos

### RQ1 (EDA)

No painel mensal 2016-01→2026-03 (119 meses, *inner join* estoque×arrecadação):

- **Escalas distintas:** dual-axis mantido; estoque $\gg$ fluxo mensal.
- **Sazonalidade:** perfil médio por mês-do-ano motiva o baseline *lag*-12.
- **Outliers IQR** destacados no scatter estoque×arrecadação — **validar com domínio jurídico; não causalizar**.
- **Composição:** ICMS (e IPVA se presente) em facet/escala separados.
- Correlação linear em níveis permanece fraca neste recorte agregado; co-movimento **não** é prova causal.

### RQ2 (forecast)

Holdout dos **últimos 12 meses**, $y=$ `valor_total`. Métricas deste run (fonte: `.local/scratch/monitoramento_faseA_metrics.json`):

| Modelo | MAPE | SMAPE | RMSE (R$) |
|---|---:|---:|---:|
| M4_Prophet | 0.1103 | 0.1154 | 1.01e+08 |
| M3_HGB_tuned | 0.1230 | 0.1334 | 1.12e+08 |
| M0_naive_sazonal_lag12 | 0.1535 | 0.1354 | 1.32e+08 |
| M0_naive_lag1 | 0.1597 | 0.1609 | 1.32e+08 |
| M1_Ridge_lags | 0.2028 | 0.2326 | 1.49e+08 |
| M1_OLS_lags | 0.2028 | 0.2326 | 1.49e+08 |
| M2_SARIMAX(0, 1, 1)x(0, 1, 1, 12) | 0.2599 | 0.2270 | 1.52e+08 |
| M5_MLP2_torch | 0.3906 | 0.4976 | 2.64e+08 |

**Melhor MAPE:** **M4_Prophet** ≈ **11.0%** (SMAPE ≈ 11.5%, RMSE ≈ 1.01e+08).

**Leitura cautelosa.** Comparar sempre ao baseline sazonal (FPP3). AIC no treino (SARIMAX) **não** implica vitória OOS. MLP-2 é *smoke* de DL — gate para DL profundo permanece fechado se M0–M4 forem competitivos. Figuras A–B mostram **todos** os modelos (barras + série).

**Não significa:** produção; causalidade estoque→arrecadação; validade fora de `extracao=2026-03`; cobertura nominal dos PI empíricos.

### Intervalos de previsão

- Prophet / SARIMAX: PI **nativo**.
- HGB / MLP: PI **empírico** (quantis OOS).

### Próximos passos

1. Painel versionado com manifesto (hash, contagens, `extracao_ref`).
2. Multi-horizonte 1/3/6 meses.
3. Fase B: `gare_janelas_mensais_v0.ipynb`.
4. Gate DL só se M0–M5 falharem de modo estável (FPP3).

### Não-afirmamos

- Não causalidade estoque → arrecadação.
- Não cobertura de dívida não inscrita / garantias.
- Não realtime nem validade fora de `extracao=2026-03`.
- Não superioridade metodológica definitiva do modelo vencedor; ranking **condicional** a este holdout.


## Referências

- Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice* (3rd ed.). OTexts. https://otexts.com/fpp3/ — accuracy (§5.8), naive (§5.2), TSCV/walk-forward (§5.10), prediction intervals (§5.5), ARIMA/SARIMA.
- Box, G. E. P., & Jenkins, G. M. (1970). *Time Series Analysis: Forecasting and Control*. Holden-Day. (tradição ARIMA/SARIMA; ver também edições posteriores Box–Jenkins–Reinsel).
- Hoerl, A. E., & Kennard, R. W. (1970). Ridge regression: Biased estimation for nonorthogonal problems. *Technometrics*, 12(1), 55–67. https://doi.org/10.1080/00401706.1970.10488634
- Friedman, J. H. (2001). Greedy function approximation: A gradient boosting machine. *Annals of Statistics*, 29(5), 1189–1232. https://doi.org/10.1214/aos/1013203451 (base do GBM; HGB via documentação scikit-learn `HistGradientBoostingRegressor`).
- Taylor, S. J., & Letham, B. (2018). Forecasting at scale. *The American Statistician*, 72(1), 37–45. https://doi.org/10.1080/00031305.2017.1380080 (Prophet).
- Tukey, J. W. (1977). *Exploratory Data Analysis*. Addison-Wesley. (regra IQR 1.5).
- Armstrong, J. S., & Collopy, F. (1992). Error measures for generalizing about forecasting methods: Empirical comparisons. *International Journal of Forecasting*, 8(1), 69–80. https://doi.org/10.1016/0169-2070(92)90008-W (contexto MAPE).
- Makridakis, S., Spiliotis, E., & Assimakopoulos, V. (2020). The M4 Competition: 100,000 time series and 61 forecasting methods. *International Journal of Forecasting*, 36(1), 54–74. https://doi.org/10.1016/j.ijforecast.2019.04.014 (uso de sMAPE em competições M; citar via FPP3 §5.8 se detalhe de fórmula divergir).
- Premissa dump: `docs/planos/00_premissa_dump.md` / `docs/api-dump/`
- Plano monitoramento: `projects/monitoramento/docs/B_monitoramento_arrecadacao.md`
- Grounded refs SoT: `docs/references/referencias_grounded.md` (front pointer: `projects/monitoramento/references/referencias_grounded.md`)
- Plotly: https://plotly.com/python/ · Kaleido (export estático): https://github.com/plotly/Kaleido

---

*Conteúdo, experimentos e conclusões são do autor; a formatação do texto teve assistência de IA.*
